# Kata 19 — Investigación Adaptativa

> Spec: [`specs/019-adaptive-investigation/spec.md`](../../specs/019-adaptive-investigation/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(model="claude-sonnet-4-6", budget_calls=12)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

En dominio desconocido, el agente NO planifica al detalle al minuto 0. Mapea topología (ls + grep), prioriza, profundiza sólo en los priorizados, y **re-planifica** cuando un hallazgo invalida el plan vigente. Todo dentro de un presupuesto acotado.

## 2. Por qué importa

Un plan rígido en territorio desconocido garantiza desperdicio: el agente sigue ramas muertas porque "estaban en el plan". Adaptar dirige atención hacia lo que la realidad muestra.

## 3. Ejemplo correcto — topology → prioritize → deep-dive → re-plan

In [ ]:
import json

# Repo simulado
FAKE_REPO = {
    "src/main.py": "from .auth import login\nfrom .billing import refund\n",
    "src/auth/__init__.py": "from .login import *\n",
    "src/auth/login.py": "def login(): pass # TODO unsafe compare",
    "src/billing/refund.py": "def refund(): pass # FIXME limit not enforced",
    "src/billing/__init__.py": "",
    "tests/test_auth.py": "def test_login(): pass",
    "docs/README.md": "# proyecto",
}

def list_files(args):
    pat = args.get("pattern", "")
    if pat:
        import fnmatch
        return [p for p in FAKE_REPO if fnmatch.fnmatch(p, pat)]
    return list(FAKE_REPO.keys())

def grep(args):
    needle = args["needle"]
    return [{"path": p, "line": (FAKE_REPO[p].split("\n").index(l)+1), "preview": l}
            for p in FAKE_REPO for l in FAKE_REPO[p].split("\n") if needle in l]

def read_file(args):
    return {"path": args["path"], "content": FAKE_REPO.get(args["path"], None)}

TOOLS = [
    {"name":"list_files","description":"Lista archivos. pattern opcional (glob).","input_schema":{"type":"object","properties":{"pattern":{"type":"string"}},"required":[]}},
    {"name":"grep","description":"Busca string en el repo.","input_schema":{"type":"object","properties":{"needle":{"type":"string"}},"required":["needle"]}},
    {"name":"read_file","description":"Lee un archivo.","input_schema":{"type":"object","properties":{"path":{"type":"string"}},"required":["path"]}},
]
HANDLERS = {"list_files": list_files, "grep": grep, "read_file": read_file}

In [ ]:
class Budget:
    def __init__(self, max_calls=8): self.max = max_calls; self.used = 0
    def take(self):
        self.used += 1
        if self.used > self.max:
            raise RuntimeError(f"presupuesto agotado ({self.max} llamadas)")

def adaptive_loop(client, *, system, user_msg, budget):
    messages = [{"role":"user","content": user_msg}]
    log = []
    while True:
        budget.take()
        resp = client.messages.create(system=system, messages=messages, tools=TOOLS)
        if resp.stop_reason == "end_turn":
            log.append({"branch":"halt"})
            return resp, log
        if resp.stop_reason != "tool_use":
            log.append({"branch":"halt", "cause": resp.stop_reason})
            return resp, log
        tu = next(b for b in resp.content if b.type == "tool_use")
        result = HANDLERS[tu.name](tu.input)
        log.append({"tool": tu.name, "args": tu.input, "n_results": (len(result) if isinstance(result, list) else 1)})
        messages.append({"role":"assistant","content": resp.content})
        messages.append({"role":"user","content":[{"type":"tool_result","tool_use_id": tu.id, "content": json.dumps(result)}]})

system = (
    "Investigas un repo desconocido buscando vulnerabilidades. Estrategia: "
    "1) lista archivos para ver topología; 2) grep TODO/FIXME/unsafe para priorizar; "
    "3) lee solo los archivos priorizados; 4) si hallazgos contradicen tu plan, re-planifica. "
    "Cierra con un resumen de hallazgos clave. PRESUPUESTO LIMITADO."
)
budget = Budget(max_calls=8)
final, log = adaptive_loop(client, system=system, user_msg="Encuentra los TODOs/FIXMEs y da un breve veredicto.", budget=budget)
for e in log: print(e)
print("\n=== veredicto final ===")
print(next((b.text for b in final.content if b.type == "text"), "")[:600])
print(f"\nllamadas usadas: {budget.used}/{budget.max}")

Observa cómo el agente intercala `list_files` / `grep` / `read_file` en función de los hallazgos previos, sin un plan rígido de inicio.

## 4. Anti-patrón — leer todo desde el inicio

In [ ]:
def naive_read_all(client, *, user_msg):
    big_dump = "\n\n".join(f"=== {p} ===\n{c}" for p, c in FAKE_REPO.items())
    sys = "Eres un investigador. Identifica vulnerabilidades."
    return client.messages.create(system=sys, messages=[{"role":"user","content": f"{user_msg}\n\nREPO:\n{big_dump}"}])

bad = naive_read_all(client, user_msg="Encuentra los TODOs/FIXMEs.")
print(next((b.text for b in bad.content if b.type == "text"), "")[:500])
print("input_tokens:", bad.usage.input_tokens)

En 7 archivos, el dump cabe. En 700, ya no — y con 7000 imposible. Adaptive loop escala; "leer todo" no.

Además, sin presupuesto explícito, el agente naive puede entrar en bucles de exploración cuando lo aplicas a un repo real con cientos de matches.

## 5. Argumento de certificación

- **Topology first, deep-dive second.** Mapeo barato (`list_files`, `grep`) antes de leer contenido.
- **Presupuesto explícito.** `Budget(max_calls=N)` evita exploración no acotada.
- **Re-planificación basada en hallazgos.** Si el agente ve algo que invalida la hipótesis, su próximo turno se ajusta.
- **Conexión con otros katas**: cada deep-dive es un sub-Kata 4 (subagente focalizado); los hallazgos persistentes van al scratchpad (Kata 18); el plan markdown puede pasar por Plan Mode (Kata 7) antes de pasar a write.

## 6. Auto-evaluación

**1. ¿Cómo decido si un hallazgo invalida el plan o sólo lo refina?**

Refina si es un detalle dentro del scope actual ("la función está en `auth.py` no en `utils.py`"). Invalida si el scope cambia ("el problema no es auth, es el middleware"). El umbral es práctico: si requiere mirar archivos no en el plan inicial, es invalidación → re-plan explícito.

**2. ¿Qué hago cuando se acaba el presupuesto sin conclusión clara?**

Devuelvo lo que tengo + flag `incomplete=true` + lista de pendientes para escalar. Nunca hago una conclusión inventada por presión de presupuesto.

**3. ¿Cómo evito loops de re-planificación?**

Cada re-plan consume del presupuesto. Si en N pasos no he reducido el espacio de exploración, abort. En código: contar re-plans y limitar a 2-3.